# 04 Customer Segmentation

This notebook turns customer-level retention metrics into operational lifecycle segments for CRM strategy and dashboarding.

Inputs:

- `../data/processed/customer_metrics.parquet`
- `../data/processed/clean_transactions.parquet`

Outputs:

- `../data/processed/customer_segments.csv`
- `../data/processed/customer_segment_summary.csv`


## Setup

The segmentation uses one mutually exclusive `primary_segment` plus supporting flags for high-value, at-risk, and lapsed status. This keeps the output easy to use in dashboards while preserving useful targeting detail.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path(os.environ.get("PROCESSED_DIR", "../data/processed"))
CUSTOMER_METRICS_PATH = PROCESSED_DIR / "customer_metrics.parquet"
TRANSACTIONS_PATH = PROCESSED_DIR / "clean_transactions.parquet"

for path in [CUSTOMER_METRICS_PATH, TRANSACTIONS_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input not found: {path}. Run notebooks 01 and 02 first."
        )

## Load Customer and Order Inputs

Customer metrics provide the base customer layer. Transaction data is regrouped to invoice level to calculate second-purchase timing and lifecycle status.

In [ ]:
customer_metrics = pd.read_parquet(CUSTOMER_METRICS_PATH)
transactions = pd.read_parquet(TRANSACTIONS_PATH)

transactions.columns = (
    transactions.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
transactions = transactions.rename(
    columns={
        "invoiceno": "invoice_no",
        "invoicedate": "invoice_date",
        "customerid": "customer_id",
    }
)
transactions["invoice_date"] = pd.to_datetime(transactions["invoice_date"])

valid_lines = transactions[transactions["is_valid_order_line"]].copy()
customer_orders = (
    valid_lines.groupby(["customer_id", "invoice_no"], as_index=False)
    .agg(
        order_date=("invoice_date", "min"),
        order_revenue=("analytical_revenue", "sum"),
    )
)
customer_orders = customer_orders[customer_orders["order_revenue"] > 0].copy()
customer_orders = customer_orders.sort_values(
    ["customer_id", "order_date", "invoice_no"]
).reset_index(drop=True)
customer_orders["order_rank"] = (
    customer_orders.groupby("customer_id").cumcount() + 1
)

analysis_date = customer_orders["order_date"].max()
print(f"Analysis date: {analysis_date.date()}")
print(f"Customers: {len(customer_metrics):,}")
print(f"Orders: {len(customer_orders):,}")

## Build Segmentation Features

The feature layer adds recency, customer revenue decile, second purchase timing, and CRM eligibility flags.

In [ ]:
second_orders = customer_orders[customer_orders["order_rank"] == 2].copy()
first_orders = customer_orders[customer_orders["order_rank"] == 1][
    ["customer_id", "order_date"]
].rename(columns={"order_date": "first_order_date"})

days_to_second = second_orders[["customer_id", "order_date"]].rename(
    columns={"order_date": "second_order_date"}
).merge(first_orders, on="customer_id", how="left")
days_to_second["days_to_second_purchase"] = (
    days_to_second["second_order_date"] - days_to_second["first_order_date"]
).dt.days

segments = customer_metrics.merge(
    days_to_second[["customer_id", "days_to_second_purchase"]],
    on="customer_id",
    how="left",
)
segments["recency_days"] = (analysis_date - segments["last_purchase"]).dt.days
segments["days_since_first_purchase"] = (
    analysis_date - segments["first_purchase"]
).dt.days

segments = segments.sort_values("revenue", ascending=False).reset_index(drop=True)
segments["revenue_rank"] = range(1, len(segments) + 1)
segments["customer_decile"] = pd.qcut(
    segments["revenue_rank"],
    10,
    labels=[
        "Top 10%",
        "10-20%",
        "20-30%",
        "30-40%",
        "40-50%",
        "50-60%",
        "60-70%",
        "70-80%",
        "80-90%",
        "Bottom 10%",
    ],
)

segments["is_high_value"] = segments["customer_decile"].eq("Top 10%")
segments["is_one_time_customer"] = segments["total_orders"].eq(1)
segments["is_early_repeater"] = segments["days_to_second_purchase"].le(30)
segments["is_delayed_repeater"] = segments["days_to_second_purchase"].gt(30)
segments["is_at_risk"] = segments["recency_days"].between(60, 89)
segments["is_lapsed"] = segments["recency_days"].ge(90)
segments["is_second_purchase_accelerator_eligible"] = (
    segments["is_one_time_customer"]
    & segments["days_since_first_purchase"].ge(21)
    & segments["recency_days"].lt(90)
)

segments.head()

## Assign Primary Segment

The priority order favours high commercial risk and clear CRM actionability: lapsed, at-risk, high-value repeat, accelerator eligible, delayed repeater, early repeater, then lower priority one-time and low-value inactive groups.

In [ ]:
segments["primary_segment"] = np.select(
    [
        segments["is_lapsed"] & segments["is_high_value"],
        segments["is_lapsed"],
        segments["is_at_risk"] & segments["is_high_value"],
        segments["is_at_risk"],
        segments["is_high_value"] & segments["is_repeat_customer"],
        segments["is_second_purchase_accelerator_eligible"],
        segments["is_delayed_repeater"],
        segments["is_early_repeater"],
        segments["is_one_time_customer"],
    ],
    [
        "High-Value Lapsed",
        "Lapsed Customer",
        "High-Value At-Risk",
        "At-Risk Customer",
        "High-Value Repeat Customer",
        "Second Purchase Accelerator Eligible",
        "Delayed Repeater",
        "Early Repeater",
        "One-Time Buyer",
    ],
    default="Low-Value Active Customer",
)

segments["recommended_crm_action"] = segments["primary_segment"].map(
    {
        "High-Value Lapsed": "Priority winback with high-touch messaging",
        "Lapsed Customer": "Low-cost winback or reactivation test",
        "High-Value At-Risk": "Priority re-engagement before lapse",
        "At-Risk Customer": "Reminder or replenishment trigger",
        "High-Value Repeat Customer": "VIP retention or loyalty treatment",
        "Second Purchase Accelerator Eligible": "Day 21 second purchase accelerator",
        "Delayed Repeater": "Cadence acceleration and replenishment flow",
        "Early Repeater": "Reinforce repeat behaviour and cross-sell",
        "One-Time Buyer": "Monitor until accelerator eligibility",
        "Low-Value Active Customer": "Low-cost nurture and product discovery",
    }
)

segments[["customer_id", "primary_segment", "recommended_crm_action"]].head()

## Segment Summary

The segment summary quantifies customer count, revenue contribution, average recency, and average order behaviour by primary segment.

In [ ]:
segment_summary = (
    segments.groupby("primary_segment", observed=False)
    .agg(
        customers=("customer_id", "count"),
        revenue=("revenue", "sum"),
        avg_revenue=("revenue", "mean"),
        avg_orders=("total_orders", "mean"),
        avg_recency_days=("recency_days", "mean"),
        accelerator_eligible_customers=(
            "is_second_purchase_accelerator_eligible",
            "sum",
        ),
    )
    .reset_index()
)
segment_summary["customer_pct"] = (
    segment_summary["customers"] / segment_summary["customers"].sum()
)
segment_summary["revenue_pct"] = (
    segment_summary["revenue"] / segment_summary["revenue"].sum()
)
segment_summary = segment_summary.sort_values(
    "revenue", ascending=False
).reset_index(drop=True)

segment_summary

## Save Outputs

The customer-level file supports CRM targeting and customer-level dashboard cuts. The summary file supports executive reporting and segment sizing.

In [ ]:
output_columns = [
    "customer_id",
    "primary_segment",
    "recommended_crm_action",
    "total_orders",
    "revenue",
    "aov",
    "first_purchase",
    "last_purchase",
    "active_days",
    "recency_days",
    "days_since_first_purchase",
    "days_to_second_purchase",
    "customer_decile",
    "is_repeat_customer",
    "is_high_value",
    "is_one_time_customer",
    "is_early_repeater",
    "is_delayed_repeater",
    "is_at_risk",
    "is_lapsed",
    "is_second_purchase_accelerator_eligible",
]

segments[output_columns].to_csv(
    PROCESSED_DIR / "customer_segments.csv",
    index=False,
)
segment_summary.to_csv(
    PROCESSED_DIR / "customer_segment_summary.csv",
    index=False,
)

print(f"Saved customer segments: {len(segments):,} customers")
print(f"Saved segment summary: {len(segment_summary):,} segments")

## Output Validation

The final checks confirm the expected output files exist and contain required columns.

In [ ]:
expected_outputs = {
    "customer_segments.csv": {
        "customer_id",
        "primary_segment",
        "recommended_crm_action",
        "recency_days",
        "customer_decile",
        "is_second_purchase_accelerator_eligible",
    },
    "customer_segment_summary.csv": {
        "primary_segment",
        "customers",
        "revenue",
        "customer_pct",
        "revenue_pct",
    },
}

for file_name, expected_columns in expected_outputs.items():
    path = PROCESSED_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"Expected output was not created: {path}")

    output = pd.read_csv(path)
    missing_columns = sorted(expected_columns - set(output.columns))
    if missing_columns:
        raise ValueError(f"{file_name} is missing columns: {missing_columns}")

    print(f"Validated {file_name}: {output.shape}")